# Evaluation Results Viewer
Reads `eval_results.json` (produced by `tools/test_tracked.py`) and presents the data as pandas DataFrames.

In [ ]:
import json
import os
import pandas as pd

# ── Configuration ────────────────────────────────────────────────────────────
RESULTS_FILE = os.path.join(os.path.dirname(os.getcwd()), 'eval_results.json')
# Override RESULTS_FILE here if your JSON lives somewhere else, e.g.:
# RESULTS_FILE = '/absolute/path/to/eval_results.json'

# ── Load & flatten ───────────────────────────────────────────────────────────
with open(RESULTS_FILE) as f:
    raw = json.load(f)

records = []
for model_name, variants in raw.items():
    for variant, runs in variants.items():
        for run in runs:
            record = {
                'model': model_name,
                'variant': variant,
                'timestamp': pd.Timestamp(run['timestamp']),
                'config': run.get('config', ''),
                'checkpoint': run.get('checkpoint', ''),
            }
            record.update(run.get('metrics', {}))
            records.append(record)

df = pd.DataFrame(records)

# Derive all metric columns (everything after the fixed columns)
ID_COLS = ["model", "variant", "timestamp"]
META_COLS = ['config', 'checkpoint']
METRIC_COLS = [c for c in df.columns if c not in (ID_COLS + META_COLS)]

print(f'Loaded {len(df)} evaluation run(s) | metrics: {METRIC_COLS}')

## All evaluation entries

In [ ]:
display_cols = ID_COLS + METRIC_COLS

all_entries = (
    df[display_cols]
    .sort_values(['model', 'variant', 'timestamp'])
    .reset_index(drop=True)
)

display(
    all_entries.style
    .format({c: '{:.4f}' for c in METRIC_COLS}, na_rep='—')
    .set_caption('All evaluation runs')
    .set_table_styles([{'selector': 'caption',
                        'props': [('font-size', '14px'), ('font-weight', 'bold')]}])
)

## Latest result per model / variant

In [ ]:
DISPLAY_COLS = ["model", "variant"] + METRIC_COLS

AP_AR_COLS = [c for c in METRIC_COLS if '/AP' in c or '/AR' in c]
OTHER_METRIC_COLS = [c for c in METRIC_COLS if c not in AP_AR_COLS]

fmt = {c: (lambda x: f'{x*100:.1f}' if pd.notna(x) else '—') for c in AP_AR_COLS}
fmt.update({c: '{:.4f}' for c in OTHER_METRIC_COLS})

latest = (
    df.sort_values('timestamp')
    .groupby(['model', 'variant'], sort=False)
    .last()
    .reset_index()
)[DISPLAY_COLS].sort_values(['model', 'variant']).reset_index(drop=True)

display(
    latest.style
    .format(fmt, na_rep='—')
    .set_caption('Latest result per model / variant')
    .set_table_styles([{'selector': 'caption',
                        'props': [('font-size', '14px'), ('font-weight', 'bold')]}])
)

## Best `coco/AP` run per model

In [ ]:
AP_COL = 'coco/AP'

if AP_COL not in df.columns:
    print(f"Column '{AP_COL}' not found in results. "
          "Available metric columns:", METRIC_COLS)
else:
    best_ap = (
        df.dropna(subset=[AP_COL])
        .sort_values(AP_COL, ascending=False)
        .groupby('model', sort=False)
        .first()
        .reset_index()
    )[ID_COLS + META_COLS + METRIC_COLS].sort_values('model').reset_index(drop=True)

    display(
        best_ap.style
        .format({c: '{:.4f}' for c in METRIC_COLS}, na_rep='—')
        .highlight_max(subset=[AP_COL], color='#c6efce')
        .set_caption(f'Best {AP_COL} run per model (variant + timestamp shown)')
        .set_table_styles([{'selector': 'caption',
                            'props': [('font-size', '14px'), ('font-weight', 'bold')]}])
    )